# Quantum ML Drug Discovery: end-to-end demonstration

This notebook walks through the portfolio workflow:

**SMILES → 3D conformers → xTB → DFT → Δ-learning → equivariant ML → forces → uncertainty → active learning → PARP1 case study.**

The notebook deliberately does **not** fabricate benchmark results. Expensive QM cells are opt-in and should be executed on the machine or compute environment used for the portfolio benchmark.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qmldd.io import load_config

cfg = load_config("../configs/quick.yaml")
seed = pd.read_csv("../data/seed/molecules.csv")
seed.head()


## 1. Generate chemically plausible conformers

RDKit ETKDG generates 3D conformers, followed by optional MMFF94s minimisation. Multiple conformers are retained because the learning target is a potential-energy surface, not one 2D molecular graph.


In [ ]:
RUN_CONFORMERS = False
if RUN_CONFORMERS:
    from qmldd.conformers import generate_conformers
    manifest = generate_conformers(
        "../data/seed/molecules.csv",
        "../data/conformers",
        n_conformers=cfg["conformers"]["n_conformers"],
        prune_rms_thresh=cfg["conformers"]["prune_rms_thresh"],
        optimize_mmff=cfg["conformers"]["optimize_mmff"],
        seed=cfg["seed"],
    )
else:
    manifest_path = Path("../data/conformers/conformer_manifest.csv")
    manifest = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame()
manifest.head()


## 2. Generate paired low- and high-fidelity quantum labels

The key physical target is

\[
\Delta E = E_{\mathrm{DFT}} - E_{\mathrm{xTB}}.
\]

Forces are handled consistently as

\[
\Delta \mathbf F = \mathbf F_{\mathrm{DFT}} - \mathbf F_{\mathrm{xTB}},
\]

so the corrected force is

\[
\mathbf F_{\mathrm{pred}} = \mathbf F_{\mathrm{xTB}} + \Delta \mathbf F_{\mathrm{ML}}.
\]

This distinction matters because differentiating the learned energy correction yields the **force correction**, not the entire DFT force.


In [ ]:
RUN_QM = False
if RUN_QM and len(manifest):
    from qmldd.qm.xtb_runner import run_xtb
    from qmldd.qm.pyscf_runner import run_pyscf

    row = next(manifest[manifest.status == "ok"].itertuples(index=False))
    xtb_result = run_xtb(row.xyz_path, row.charge, row.multiplicity)
    dft_result = run_pyscf(row.xyz_path, row.charge, row.multiplicity, functional="b3lyp", basis="def2-svp")
    print("xTB eV:", xtb_result.energy_ev)
    print("DFT eV:", dft_result.energy_ev)
    print("Delta eV:", dft_result.energy_ev - xtb_result.energy_ev)


## 3. Build a leakage-resistant dataset

Conformers of the same molecule must not be split across training and test sets. The quick configuration uses molecule-level splitting; the production configuration uses Bemis-Murcko scaffold splitting to create a harder OOD challenge.


In [ ]:
paired_path = Path("../data/processed/paired.csv")
if paired_path.exists():
    paired = pd.read_csv(paired_path)
    display(paired[["molecule_id", "conformer_id", "xtb_energy_ev", "dft_energy_ev", "delta_energy_ev", "split"]].head())
    display(paired.groupby("split")["molecule_id"].nunique())
else:
    print("Run scripts/03_build_dataset.py after the QM calculations.")


## 4. Train the E(3)-equivariant Δ-energy model

The e3nn model represents angular information with spherical-harmonic irreducible representations. The final energy correction is a scalar, while force corrections are derived through automatic differentiation.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from qmldd.train import train_ensemble
    train_ensemble(cfg, "../data/processed/paired.csv")


## 5. Benchmark against the xTB baseline

A useful portfolio result is not merely a small test MAE. The benchmark should answer several questions:

- Does Δ-learning improve the xTB baseline?
- Does it preserve conformer ranking?
- Are force corrections accurate?
- Does uncertainty increase on difficult or OOD molecules?
- What is the measured speed gain over DFT on identical hardware?
- How much does performance degrade on scaffold-held-out chemistry?


In [ ]:
pred_path = Path("../results/test_predictions.csv")
if pred_path.exists():
    pred = pd.read_csv(pred_path)
    pred["abs_xtb_error"] = (pred.xtb_energy_ev - pred.dft_energy_ev).abs()
    pred["abs_ml_error"] = (pred.pred_dft_ev - pred.dft_energy_ev).abs()
    display(pred.groupby("molecule_id")[["abs_xtb_error", "abs_ml_error", "uncertainty_ev"]].mean())

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(pred.uncertainty_ev, pred.abs_ml_error)
    ax.set_xlabel("Ensemble uncertainty (eV)")
    ax.set_ylabel("Absolute ΔML error (eV)")
    ax.set_title("Uncertainty versus error")
    plt.show()
else:
    print("Run scripts/05_evaluate.py after training.")


## 6. Active learning

In a true acquisition loop, the candidate pool has xTB labels but **not yet DFT labels**. Ensemble uncertainty is used to decide which conformers should receive the next expensive DFT calculations. Test-set labels should never be used to make acquisition decisions in a reported benchmark.


In [ ]:
from qmldd.active_learning import select_by_uncertainty

if pred_path.exists():
    # Demonstration only. For a real loop, replace this with predictions from an unlabelled pool.
    demo_acquisition = select_by_uncertainty(
        pred[["molecule_id", "conformer_id", "uncertainty_ev"]], acquire_n=min(5, len(pred))
    )
    display(demo_acquisition)


## 7. PARP1 translational case study

Known PARP inhibitors provide a biologically meaningful ligand panel for conformational energetics and OOD testing. The correct claim is modest: the model can be used to study **intramolecular energetics and uncertainty** for these ligands.

It should not be presented as a direct model of PARP1 binding affinity, PARP trapping, synthetic lethality, or clinical response without additional protein, solvent, sampling, biochemical, and cellular modelling.


In [ ]:
parp = pd.read_csv("../data/seed/parp1_ligands.csv")
display(parp[["name", "biological_note"]])


## conclusion

The central story is:

> A fast semi-empirical method supplies most of the physical signal, expensive DFT calculations define higher-fidelity corrections, an equivariant graph network learns those corrections, force consistency is maintained through energy gradients, and uncertainty decides where additional QM compute is most valuable.

That is a stronger scientific narrative than simply training another molecular-property predictor.
